# Controlled register addition

Copyright 2024 Google LLC. Adapted under Apache-2.0 from locked Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

import torch
from IPython import display as notebook_display

from examples.algorithms.addition_catalog import addition_signature_svg
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_generalization import (
    ConjunctionCall,
    ResourceCall,
)
from flagquantum.compiler.registers.arithmetic.arithmetic_declarations import (
    arithmetic_call_graph,
)
from flagquantum.compiler.registers.arithmetic.controlled_add_resource_model import (
    source_controlled_add_resource_formula,
)
from flagquantum.compiler.registers.arithmetic.controlled_addition import (
    controlled_add_adjoint_ir,
    controlled_add_ir,
)
from flagquantum.core.ir.program.register_types import (
    quantum_bit_type,
    unsigned_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic

# Optional IPython functions enter through the presentation boundary.
SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def call_title(call: ResourceCall) -> str:
    if isinstance(call, ArithmeticCall):
        label = "Controlled add" if call.operation == "controlled_add" else "Add"
        return f"{label} ({call.widths[0]} bits)"
    if isinstance(call, ConjunctionCall):
        return "And adjoint" if call.adjoint else "And"
    return str(call)


## Controlled addition
An n-bit controlled-addition gate.

This uses $n$ `And` gates plus an uncontrolled adder for a total of $2n - 1$ Toffolis.

#### Parameters
 - `a_dtype`: Quantum datatype used to represent the integer a.
 - `b_dtype`: Quantum datatype used to represent the integer b. Must be large enough to hold the result in the output register of a + b, or else it simply drops the most significant bits. If not specified, b_dtype is set to a_dtype.
 - `cv`: When controlled=0, this bloq is active when the ctrl register is 0. When controlled=1, this bloq is active when the ctrl register is 1. 

#### Registers
 - `ctrl`: the control bit for the addition
 - `a`: A a_dtype.bitsize-sized input register (register a above).
 - `b`: A b_dtype.bitsize-sized input/output register (register b above). 

#### References
 - [Halving the cost of quantum addition](https://arxiv.org/abs/1709.06648). 


In [ ]:
def example(width: int) -> tuple[ArithmeticCall, Signature]:
    return ArithmeticCall("controlled_add", (width, width)), Signature(
        (
            Register("ctrl", quantum_bit_type()),
            Register("a", unsigned_register_type(width)),
            Register("b", unsigned_register_type(width)),
        )
    )


### Example Instances

In [ ]:
cadd_small = example(3)


In [ ]:
cadd_large = example(1000)


#### Graphical Signature

In [ ]:
display(
    SVG(
        addition_signature_svg(
            [
                (name, call, signature)
                for name, (call, signature) in (
                    ("small", cadd_small),
                    ("large", cadd_large),
                )
            ]
        )
    )
)


### Call Graph

In [ ]:
root: ResourceCall = cadd_small[0]
cadd_graph: SymbolicCallGraph[ResourceCall] = symbolic_call_graph(
    root, arithmetic_call_graph(cadd_small[0]), max_depth=1
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                cadd_graph.root, cadd_graph.direct_calls, title=call_title
            )
        )
    )
)
display(cadd_graph.leaf_counts)


## Coherent append execution and separate resource accounts

Both original instance widths and the small one-level call graph are retained. The original signature display and call graph execute through their signature-only drawing route. Native signatures render through the existing table owner. The separate full musical-score route retains the source callback signature error recorded in the complete source audit.

The circuit below acts on every three-bit input and a coherent spectator. Five explicit work qubits start and end in zero. Its exact adjoint restores the full complex state. Primitive inventory and the independent model are displayed separately. At 1000 bits the original peak-width query reaches its recursive Add decomposition and fails at the normal recursion limit, while its declared Toffoli model remains available.


In [ ]:
left, right = (1, 2, 3), (4, 5, 6)
loaded, carries = (8, 9, 10), (11, 12)
load_bits, carry_bits = (0, 1, 2), (3, 4)
gates = controlled_add_ir(0, left, right, loaded, carries, load_bits, carry_bits)
coefficients = torch.arange(256, dtype=torch.float64)
amplitudes = torch.complex(coefficients + 1, 3 * coefficients - 5)
amplitudes /= amplitudes.norm()
initial = torch.zeros(1 << 13, dtype=torch.complex128)
expected = torch.zeros_like(initial)
for raw, amplitude in enumerate(amplitudes):
    control, a, b, spectator = raw >> 7, (raw >> 4) & 7, (raw >> 1) & 7, raw & 1
    output = (b + a) % 8 if control else b
    initial[raw << 5] = amplitude
    expected[(((control << 7) | (a << 4) | (output << 1) | spectator) << 5)] = amplitude
circuit = DynamicCircuit(13, inputs=initial, dtype=torch.complex128)
circuit.append(FieldOperation(13, gates))
result = run_dynamic(circuit, shots=8, seed=41, strategy="trajectory")
torch.testing.assert_close(
    result.final_states.reshape(8, -1), expected.expand(8, -1), rtol=0, atol=2e-12
)
inverse = controlled_add_adjoint_ir(
    0, left, right, loaded, carries, load_bits, carry_bits
)
circuit.append(FieldOperation(13, inverse))
restored = run_dynamic(circuit, shots=8, seed=41, strategy="trajectory")
torch.testing.assert_close(
    restored.final_states.reshape(8, -1), initial.expand(8, -1), rtol=0, atol=2e-12
)
measured_primitives = dict(Counter(g.name for g in gates))
modeled_resources = source_controlled_add_resource_formula(3, 3)
display(measured_primitives)
display(modeled_resources)
